# Aula 10 — Como Funciona um LLM

Três exercícios, um depois de cada microaula. Rode as células na ordem.
Troque cada `____` pelo que falta e rode; a célula seguinte confere: ✅
deu certo, ❌ traz uma pista. As células 🧰 são código pronto: só rode.

**Travou?** [Colas dos exercícios](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios).

## Preparação

🧰 O tokenizador da Aula 9, o texto de Machado em tokens, e o modelo que
o professor treinou (você monta este mesmo modelo no Exercício 3). Leva
uns 30 segundos.

In [ ]:
import collections
import json
import math
import os
import re
import urllib.request
import warnings

import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import ops
from sklearn.decomposition import PCA

keras.utils.set_random_seed(0)
warnings.filterwarnings("ignore", message="Skipping variable loading for optimizer")
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/"


def baixar(nome, binario=False):
    local = os.path.join("..", "..", "data", nome)   # só existe no PC do professor
    if os.path.exists(local):
        if binario:
            return open(local, "rb").read()
        return open(local, encoding="utf-8").read()
    dados = urllib.request.urlopen(URL_DADOS + nome).read()
    if binario:
        return dados
    return dados.decode("utf-8")


# --- Tokenizador (Aula 9) ---
oficial = json.loads(baixar("machado_bpe.json"))
PADRAO = re.compile(oficial["padrao"])
fusoes = {}
for k, (a, b) in enumerate(oficial["fusoes"]):
    fusoes[(a, b)] = 256 + k
tabela = {}
for i in range(256):
    tabela[i] = bytes([i])
for (a, b), novo in fusoes.items():
    tabela[novo] = tabela[a] + tabela[b]


def contar_pares(ids):
    pares = collections.Counter()
    for par in zip(ids, ids[1:]):
        pares[par] += 1
    return pares


def fundir(ids, par, novo):
    saida = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == par[0] and ids[i + 1] == par[1]:
            saida.append(novo)
            i = i + 2
        else:
            saida.append(ids[i])
            i = i + 1
    return saida


def ordem(par):
    return fusoes.get(par, 10**9)


def codificar_palavra(palavra):
    ids = list(palavra.encode("utf-8"))
    while len(ids) >= 2:
        par = min(contar_pares(ids), key=ordem)
        if par not in fusoes:
            break
        ids = fundir(ids, par, fusoes[par])
    return ids


memoria = {}


def codificar(texto):
    saida = []
    for palavra in PADRAO.findall(texto):
        if palavra not in memoria:
            memoria[palavra] = codificar_palavra(palavra)
        saida.extend(memoria[palavra])
    return saida


def decodificar(ids):
    juntos = b""
    for i in ids:
        juntos = juntos + tabela[int(i)]
    return juntos.decode("utf-8", errors="replace")


def mostrar(i):
    return decodificar([i]).replace(" ", "·")       # o espaço aparece como ·


# --- O texto em tokens ---
TAMANHO_VOCABULARIO = 1024
BLOCO = 128          # janela de contexto
LOTE = 32

tokens = np.array(codificar(baixar("machado.txt")))
corte = int(0.9 * len(tokens))
dados_treino = tokens[:corte]
dados_validacao = tokens[corte:]


def sortear_lote(dados):
    inicios = np.random.randint(0, len(dados) - BLOCO - 1, LOTE)
    entradas = []
    alvos = []
    for inicio in inicios:
        entradas.append(dados[inicio : inicio + BLOCO])
        alvos.append(dados[inicio + 1 : inicio + BLOCO + 1])
    return np.stack(entradas), np.stack(alvos)


# --- O modelo do professor ---
DIMENSAO = 128
CABECAS = 4
CAMADAS = 4
OCULTA = 256


class TokenEPosicao(keras.layers.Layer):
    """Soma o embedding de cada token com o embedding da posição dele."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.token = keras.layers.Embedding(TAMANHO_VOCABULARIO, DIMENSAO, name="token")
        self.posicao = keras.layers.Embedding(BLOCO, DIMENSAO, name="posicao")

    def call(self, ids):
        n = ops.shape(ids)[1]
        return self.token(ids) + self.posicao(ops.arange(n))


def modelo_do_professor():
    entrada = keras.Input(shape=(None,), dtype="int32", name="tokens")
    x = TokenEPosicao(name="embedding")(entrada)
    for i in range(CAMADAS):
        h = keras.layers.LayerNormalization(name=f"norma_a{i}")(x)
        h = keras.layers.MultiHeadAttention(num_heads=CABECAS, key_dim=DIMENSAO // CABECAS,
                                            name=f"atencao{i}")(h, h, use_causal_mask=True)
        x = x + h
        h = keras.layers.LayerNormalization(name=f"norma_b{i}")(x)
        h = keras.layers.Dense(OCULTA, activation="gelu", name=f"densa{i}")(h)
        x = x + keras.layers.Dense(DIMENSAO, name=f"volta{i}")(h)
    x = keras.layers.LayerNormalization(name="norma_final")(x)
    saida = keras.layers.Dense(TAMANHO_VOCABULARIO, name="saida")(x)
    modelo = keras.Model(entrada, saida)
    open("mini_llm.weights.h5", "wb").write(baixar("mini_llm.weights.h5", binario=True))
    modelo.load_weights("mini_llm.weights.h5")
    return modelo


professor = modelo_do_professor()
print(f"{len(tokens):,} tokens de texto e o modelo do professor prontos.".replace(",", "."))

---
## Exercício 1: o token vira vetor

📝 Antes: [quiz da Microaula 1](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-10-como-funciona-um-llm#microaula-1-o-token-vira-vetor).
🆘 Travou? [Cola do Exercício 10.1](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-10-1)

🎯 O exemplo de treino (entrada e alvo), o lote, e o espaço onde o
modelo guarda o sentido de cada token.

### 1.a Entrada e alvo

O alvo é a mesma frase, andada uma casa: cada token tem que prever o
seguinte.

In [ ]:
ids = codificar("O menino era pai do homem.")
entrada = ids[:-1]
alvo = ids[____:]

for e, a in zip(entrada, alvo):
    print(f"{mostrar(e):>8}  →  {mostrar(a)}")

In [ ]:
if alvo == ids[1:]:
    print(f"✅ {len(ids)} tokens de texto deram {len(alvo)} exemplos de treino, todos na mesma passada.")
else:
    print("❌ O alvo começa um token depois da entrada: ids[1:].")

### 1.b O lote passa pelo embedding

🧰 `sortear_lote(dados_treino)` sorteia 32 janelas de 128 tokens, e os
alvos de cada uma. A camada `Embedding` troca cada token pela linha dele
numa tabela: uma linha por token do vocabulário, 128 números por linha.
Quantas linhas a tabela precisa ter?

In [ ]:
entrada, alvo = sortear_lote(dados_treino)
print("entrada:", entrada.shape, "  alvo:", alvo.shape)

embedding = keras.layers.Embedding(input_dim=____, output_dim=128)
vetores = embedding(entrada)
print("depois do embedding:", vetores.shape)

In [ ]:
if tuple(vetores.shape) == (32, 128, 128) and embedding.input_dim == 1024:
    pesos = f"{embedding.count_params():,}".replace(",", ".")
    print(f"✅ 32 janelas × 128 posições × 128 números. A tabela tem {pesos} pesos, "
          "e o treino ajusta todos.")
else:
    print("❌ Uma linha para cada token do vocabulário: input_dim=TAMANHO_VOCABULARIO (1024).")

### 1.c Os vizinhos de um token, no modelo treinado

🧰 `vizinhos(palavra)` procura, na tabela de embedding do modelo do
professor, os vetores que apontam para o mesmo lado que o da palavra
(a similaridade do cosseno). Nenhuma regra de gramática foi ensinada ao
modelo. Comece a palavra com espaço, como ela aparece no meio do texto.

In [ ]:
tabela_embedding = np.asarray(professor.get_layer("embedding").token.embeddings)
direcoes = tabela_embedding / np.linalg.norm(tabela_embedding, axis=1, keepdims=True)


def vizinhos(palavra, quantos=6):
    ids = codificar(palavra)
    if len(ids) != 1:
        print(f"{palavra!r} vira {len(ids)} tokens; escolha uma palavra que seja um token só.")
        return
    similaridades = direcoes @ direcoes[ids[0]]
    ordem_ = np.argsort(similaridades)[::-1][1 : quantos + 1]
    texto = "   ".join(f"{mostrar(i)} {similaridades[i]:.2f}" for i in ordem_)
    print(f"{mostrar(ids[0]):>8}:  {texto}")


for palavra in [" elle", " dous", " era", " olhos", " quando", " Rubião"]:
    vizinhos(palavra)

✏️ Teste palavras suas, trocando a lista. Que tipo de palavra fica perto
de que tipo? →

### 1.d O espaço do sentido, em 3D

🧰 Os vetores têm 128 números. Para desenhar, a célula espreme os 128 em
3 (com uma PCA, a mesma da Aula 3) e mostra o mesmo espaço de três
ângulos. Cada cor é um grupo de palavras.

In [ ]:
grupos = {
    "números": [" dous", " tres", " duas", " alguns", " outros", " todos"],
    "pessoas": [" elle", " ella", " Rubião", " Helena", " Jorge", " moça"],
    "verbos": [" era", " foi", " fosse", " eram", " estava", " tinha"],
    "corpo": [" olhos", " mão", " mãos", " cabeça", " coração"],
    "conectivos": [" quando", " porque", " onde", " quem", " como"],
}
cores = ["#B85042", "#2B6CB0", "#2F855A", "#B7791F", "#6B46C1"]

palavras, ids, cor_de = [], [], []
for (grupo, lista), cor in zip(grupos.items(), cores):
    for palavra in lista:
        codigo = codificar(palavra)
        if len(codigo) == 1:
            palavras.append(palavra.strip())
            ids.append(codigo[0])
            cor_de.append(cor)

pontos = PCA(n_components=3).fit_transform(direcoes[ids])
figura = plt.figure(figsize=(15, 5))
for n, angulo in enumerate([20, 110, 200]):
    eixo = figura.add_subplot(1, 3, n + 1, projection="3d")
    eixo.scatter(pontos[:, 0], pontos[:, 1], pontos[:, 2], c=cor_de, s=40)
    for (x, y, z), palavra, cor in zip(pontos, palavras, cor_de):
        eixo.text(x, y, z, " " + palavra, color=cor, fontsize=9)
    eixo.view_init(elev=20, azim=angulo)
    eixo.set_xticks([]); eixo.set_yticks([]); eixo.set_zticks([])
plt.tight_layout()
plt.show()

---
## Exercício 2: a atenção

📝 Antes: [quiz da Microaula 2](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-10-como-funciona-um-llm#microaula-2-a-atencao).
🆘 Travou? [Cola do Exercício 10.2](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-10-2)

🎯 A camada de atenção do Keras, com quatro cabeças e a máscara causal, e
para onde o modelo treinado olha.

### 2.a Quatro cabeças

Os vetores têm 128 números, e cada cabeça trabalha com 32. Quantas
cabeças cabem?

In [ ]:
atencao = keras.layers.MultiHeadAttention(num_heads=____, key_dim=32)

frase = keras.random.normal((1, 6, 128))           # uma frase de 6 tokens, já em vetores
saida, pesos = atencao(frase, frase, return_attention_scores=True)
print("saída:", saida.shape, "  pesos de atenção:", pesos.shape)

In [ ]:
if tuple(pesos.shape) == (1, 4, 6, 6):
    print("✅ Cada cabeça tem a sua tabela de 6 × 6: quem olha (linha) para quem (coluna). "
          "A saída volta a ter 128 números por token.")
else:
    print("❌ 128 números divididos em cabeças de 32: num_heads=4.")

### 2.b A máscara causal

Sem máscara, cada posição enxerga o futuro, que é justamente o que ela
tem que prever. Ligue a máscara e olhe a tabela da primeira cabeça.

In [ ]:
saida, pesos = atencao(frase, frase, use_causal_mask=____, return_attention_scores=True)

print("Pesos da cabeça 1, em %  (linha = quem olha, coluna = para quem):")
print(np.round(np.asarray(pesos[0, 0]) * 100).astype(int))

In [ ]:
tabela_pesos = np.asarray(pesos[0, 0])
# o teste da última palavra: troque só o último token e veja se o passado muda
outra = ops.concatenate([frase[:, :5], keras.random.normal((1, 1, 128))], axis=1)
saida_outra = atencao(outra, outra, use_causal_mask=True)
passado_igual = np.allclose(np.asarray(saida[0, :5]), np.asarray(saida_outra[0, :5]), atol=1e-5)
if np.allclose(np.triu(tabela_pesos, k=1), 0) and passado_igual:
    print("✅ Acima da diagonal, tudo zero: ninguém olha o futuro. E trocar a última "
          "palavra não mudou nada nas cinco primeiras posições.")
else:
    print("❌ Ligue a máscara: use_causal_mask=True.")

### 2.c Para onde o modelo treinado olha

🧰 `pesos_de_atencao(texto, camada)` passa o texto pelo modelo do
professor e devolve os pesos das quatro cabeças de uma camada (de 0 a 3).
Mude a camada e compare.

In [ ]:
def pesos_de_atencao(texto, camada):
    ids = np.array([codificar(texto)])
    x = professor.get_layer("embedding")(ids)
    for i in range(camada + 1):
        h = professor.get_layer(f"norma_a{i}")(x)
        h, pesos = professor.get_layer(f"atencao{i}")(h, h, use_causal_mask=True,
                                                      return_attention_scores=True)
        x = x + h
        h = professor.get_layer(f"densa{i}")(professor.get_layer(f"norma_b{i}")(x))
        x = x + professor.get_layer(f"volta{i}")(h)
    return np.asarray(pesos[0]), [mostrar(i) for i in ids[0]]


texto = "A casa de Capitú tinha uma janela para a rua."
pesos, rotulos = pesos_de_atencao(texto, camada=0)

figura, eixos = plt.subplots(1, 4, figsize=(18, 5))
for cabeca, eixo in enumerate(eixos):
    eixo.imshow(pesos[cabeca], cmap="Oranges", vmin=0, vmax=1)
    eixo.set_xticks(range(len(rotulos)), rotulos, rotation=90, fontsize=8)
    eixo.set_yticks(range(len(rotulos)), rotulos, fontsize=8)
    eixo.set_title(f"cabeça {cabeca + 1}")
plt.tight_layout()
plt.show()

✏️ Por que o canto de cima, à direita, é todo branco? E as quatro cabeças
olham para os mesmos lugares? →

---
## Exercício 3: o modelo inteiro

📝 Antes: [quiz da Microaula 3](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-10-como-funciona-um-llm#microaula-3-o-bloco-e-o-modelo-inteiro).
🆘 Travou? [Cola do Exercício 10.3](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-10-3)

🎯 O seu LLM, montado com camadas prontas do Keras, com os mesmos 809.728
pesos do modelo do professor.

### 3.a O bloco

Duas metades, cada uma com o seu atalho: normaliza, atenção, soma;
normaliza, camada densa, soma. Duas lacunas: o tamanho em que a camada
densa volta, e o que o bloco devolve.

In [ ]:
def bloco(x, i):
    # 1ª metade: as posições conversam
    h = keras.layers.LayerNormalization(name=f"norma_a{i}")(x)
    h = keras.layers.MultiHeadAttention(num_heads=CABECAS, key_dim=DIMENSAO // CABECAS,
                                        name=f"atencao{i}")(h, h, use_causal_mask=True)
    x = x + h                                                      # atalho
    # 2ª metade: cada posição pensa sozinha
    h = keras.layers.LayerNormalization(name=f"norma_b{i}")(x)
    h = keras.layers.Dense(OCULTA, activation="gelu", name=f"densa{i}")(h)
    h = keras.layers.Dense(____, name=f"volta{i}")(h)
    return ____                                                    # atalho

In [ ]:
entrada_teste = keras.Input(shape=(None, DIMENSAO))
teste = keras.Model(entrada_teste, bloco(entrada_teste, 99))
for peso in teste.weights:                       # zera todos os pesos do bloco
    peso.assign(ops.zeros(peso.shape))
x_teste = np.random.randn(1, 5, DIMENSAO).astype("float32")
y_teste = np.asarray(teste(x_teste))
if y_teste.shape == x_teste.shape and np.allclose(y_teste, x_teste, atol=1e-5):
    print("✅ Com todos os pesos zerados, o bloco devolve a entrada intacta: é o atalho. "
          "Cada bloco só acrescenta alguma coisa ao que já estava lá.")
else:
    print("❌ A camada volta para DIMENSAO (128) números, e o bloco devolve x + h.")

### 3.b O modelo

Embedding de token e de posição, os blocos, uma normalização final e a
saída, com uma pontuação para cada token do vocabulário. Quantos blocos?
Quantas pontuações?

In [ ]:
def montar_modelo():
    entrada = keras.Input(shape=(None,), dtype="int32", name="tokens")
    x = TokenEPosicao(name="embedding")(entrada)
    for i in range(____):
        x = bloco(x, i)
    x = keras.layers.LayerNormalization(name="norma_final")(x)
    saida = keras.layers.Dense(____, name="saida")(x)
    return keras.Model(entrada, saida)


modelo = montar_modelo()
print(f"{modelo.count_params():,} pesos".replace(",", "."))

In [ ]:
if modelo.count_params() == 809_728:
    print("✅ 809.728 pesos, o mesmo número do slide. Onde eles moram:")
    grupos_de_pesos = {"embedding": 0, "atenção": 0, "camada densa": 0, "normalização": 0, "saída": 0}
    for camada in modelo.layers:
        nome = camada.name
        if nome == "embedding":
            grupos_de_pesos["embedding"] += camada.count_params()
        elif nome.startswith("atencao"):
            grupos_de_pesos["atenção"] += camada.count_params()
        elif nome.startswith(("densa", "volta")):
            grupos_de_pesos["camada densa"] += camada.count_params()
        elif nome.startswith("norma"):
            grupos_de_pesos["normalização"] += camada.count_params()
        elif nome == "saida":
            grupos_de_pesos["saída"] += camada.count_params()
    for grupo, quantos in grupos_de_pesos.items():
        print(f"   {grupo:<13} {quantos:>8,}  ({quantos / 809_728:.1%})".replace(",", "."))
else:
    pesos = f"{modelo.count_params():,}".replace(",", ".")
    print(f"❌ Saíram {pesos} pesos. São CAMADAS blocos, e a saída tem "
          "TAMANHO_VOCABULARIO pontuações.")

### 3.c O passo zero

O modelo devolve pontuações cruas (os *logits*); a perda faz a softmax
por conta própria se você avisar. Um modelo recém-montado chuta igual
entre 1.024 tokens, então a perda dele tem que dar perto de
ln(1024) = 6,93.

In [ ]:
entrada, alvo = sortear_lote(dados_validacao)      # texto que nenhum modelo viu no treino
modelo.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=____))
perda = modelo.evaluate(entrada, alvo, verbose=0)
print(f"perda: {perda:.2f}   perplexidade: {math.exp(perda):.0f}   "
      f"(o chute ao acaso dá {math.log(1024):.2f} e 1024)")

In [ ]:
if 6.5 < perda < 7.4:
    print("✅ O seu modelo ainda não sabe nada, e é isso que o número diz: "
          "ele está na dúvida entre uns mil tokens.")
else:
    print("❌ O modelo devolve logits: from_logits=True.")

### 3.d Os pesos do professor no seu modelo

🧰 Se o seu modelo é igual ao do professor, peça por peça, os pesos
treinados dele cabem no seu. A célula carrega, mede de novo no mesmo
lote de validação e deixa o seu modelo escrever. Na Aula 11, você treina os seus próprios pesos.

In [ ]:
def gerar(modelo, texto, quantos=60, temperatura=0.8):
    indices = codificar(texto)
    for _ in range(quantos):
        logits = modelo(np.array([indices[-BLOCO:]]))
        sorteado = keras.random.categorical(logits[:, -1] / temperatura, 1)
        indices.append(int(sorteado[0, 0]))
    return decodificar(indices)


modelo.load_weights("mini_llm.weights.h5")
perda_treinado = modelo.evaluate(entrada, alvo, verbose=0)
print(f"perda: {perda_treinado:.2f}   perplexidade: {math.exp(perda_treinado):.0f}\n")
for comeco in ["A casa de", "Não sei se a senhora", "Capitú"]:
    print(gerar(modelo, comeco))
    print("---")

In [ ]:
if perda_treinado < 4:
    print(f"✅ Mesmos pesos, mesma máquina: a perda caiu de {perda:.2f} para {perda_treinado:.2f}. "
          f"Com perplexidade {math.exp(perda_treinado):.0f}, o modelo escolhe entre poucas dezenas "
          "de tokens, e não mais entre 1.024.")
else:
    print("❌ Os pesos não bateram: confira o bloco e o modelo nos passos 3.a e 3.b.")

---
## O que você construiu hoje

| Peça | Função |
|---|---|
| entrada e alvo, `sortear_lote` | o texto vira exemplos de treino: 4.096 previsões por lote |
| `Embedding` | cada token vira um vetor; sentido parecido, vetor parecido |
| `MultiHeadAttention` com máscara | cada posição olha para trás, com quatro olhares |
| `bloco` e `montar_modelo` | **o seu LLM**: 809.728 pesos, pronto para treinar na Aula 11 |